##Setup: Imports and Drive Connection
We're setting up our final experiment. We're importing cv2 for preprocessing, ResNet50 and its preprocess_input function, and tf.numpy_function as per the technical feedback we received

In [ ]:
# here we are importing all the required libraries
import tensorflow as tf
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, RandomFlip, RandomRotation, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import matplotlib.pyplot as plt
import os
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import cv2

# here we are connecting to google drive
print("connecting to google drive...")
from google.colab import drive
drive.mount('/content/drive')
print("drive mounted successfully")


Connecting to Google Drive...
Mounted at /content/drive
Drive mounted successfully.


##Load Data: Unzip and Define Paths
No changes here. We're unzipping our data and setting up all our file paths and epoch counts

In [ ]:
# here we are unzipping the dataset from google drive
print("unzipping the data zip file from drive...")
ZIP_PATH = "/content/drive/MyDrive/DATA.zip"
!unzip -o -q {ZIP_PATH} -d "/content/"
print("data is unzipped and ready in content")

# here we are defining the main project variables
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS_STAGE_1 = 15
EPOCHS_STAGE_2 = 10

# here we are defining the paths to our training validation and testing folders
TRAIN_DIR = "/content/DATA/Training(70%)"
VALID_DIR = "/content/DATA/Validation(20%)"
TEST_DIR = "/content/DATA/Testing(10%)"


Unzipping the DATA.zip file from Drive...
Data is unzipped and ready in /content/.


##Define Preprocessing Functions
apply_preprocessing: This is our OpenCV function. It takes a NumPy array, converts it to uint8 (which OpenCV needs), applies the blur and CLAHE, then converts it back to float32.

tf_preprocess_wrapper: This uses tf.numpy_function to safely run our OpenCV code inside the TensorFlow pipeline

In [ ]:
# here we are defining the corrected preprocessing functions

# here we are creating a function that runs in numpy
def apply_preprocessing(image_array):
    # here we are converting the image from float32 to uint8 for opencv
    image_uint8 = image_array.astype(np.uint8)

    # here we are removing noise using median blur
    image_blur = cv2.medianBlur(image_uint8, 5)

    # here we are enhancing contrast using clahe on the l channel
    image_lab = cv2.cvtColor(image_blur, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(image_lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l_channel)

    # here we are merging the channels and converting back to rgb
    merged = cv2.merge((cl, a_channel, b_channel))
    final_image = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

    # here we are returning the processed image as float32
    return final_image.astype(np.float32)

# here we are defining a wrapper that uses the correct tf numpy function
@tf.function
def tf_preprocess_wrapper(image, label):
    # here we are using tf numpy function to apply our opencv preprocessing
    [image,] = tf.numpy_function(
        apply_preprocessing,
        [image],
        [tf.float32]
    )

    # here we are setting the image shape back to the right size
    image.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    return image, label


##Create Data Pipelines
Here, we'll load all three datasets (train, valid, test). We'll also run our class weight calculation, which we've proved is essential. Finally, we .map() our new tf_preprocess_wrapper to all three datasets to apply our blur/CLAHE, and then we optimize them with .prefetch()

In [ ]:
# here we are creating data pipelines with preprocessing
print("loading training, validation, and test datasets...")

# here we are loading the training dataset with categorical labels
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, label_mode="categorical", seed=123,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    interpolation='bilinear'
)

# here we are loading the validation dataset
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR, label_mode="categorical", seed=123,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    interpolation='bilinear'
)

# here we are loading the test dataset without shuffling
test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, label_mode="categorical", image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, shuffle=False, interpolation='bilinear'
)

# here we are checking the class names
class_names = train_dataset.class_names
print(f"our class names are: {class_names}")

# here we are calculating the class weights for balanced training
print("calculating class weights...")
train_labels = np.concatenate([np.argmax(y.numpy(), axis=1) for x, y in train_dataset])
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
print(f"calculated weights: {class_weight_dict}")

# here we are optimizing and applying preprocessing to all datasets
print("optimizing data pipelines and applying preprocessing...")
AUTOTUNE = tf.data.AUTOTUNE

# here we are unbatching, applying preprocessing, and rebatching the datasets
train_dataset = train_dataset.unbatch().map(tf_preprocess_wrapper, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.unbatch().map(tf_preprocess_wrapper, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.unbatch().map(tf_preprocess_wrapper, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)

# here we are confirming that preprocessing has been applied to all datasets
print("preprocessing function has been mapped to all datasets")


Loading Training, Validation, and Test datasets...
Found 2297 files belonging to 4 classes.
Found 573 files belonging to 4 classes.
Found 394 files belonging to 4 classes.
Our class names are: ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
Calculating class weights...
Calculated weights: {0: np.float64(0.8687594553706506), 1: np.float64(0.8727203647416414), 2: np.float64(1.817246835443038), 3: np.float64(0.8674471299093656)}
Optimizing data pipelines and applying preprocessing...
Preprocessing function has been mapped to all datasets.


## We load ResNet50.

We build our pipeline: Input -> RandomFlip -> RandomRotation (our augmentations) -> Lambda(preprocess_input) (the fix for ResNet's normalization) -> base_model (frozen).

We'll compile and run model.fit() with our class_weight_dict

In [ ]:
# here we are starting stage 1 training with advanced augmentations
print("building the resnet50 model...")

# here we are loading the resnet50 base model without the preprocessing function
base_model = ResNet50(weights='imagenet', include_top=False,
                      input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

# here we are freezing the base layers for stage 1 training
print("freezing the resnet50 base layers for stage 1")
base_model.trainable = False

# here we are defining the input shape
inputs = Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

# here we are applying advanced image augmentations before normalization
print("applying advanced augmentations flip rotation zoom and contrast...")
x = RandomFlip('horizontal')(inputs)
x = RandomRotation(0.1)(x)
x = tf.keras.layers.RandomZoom(0.1)(x)
x = tf.keras.layers.RandomContrast(0.1)(x)

# here we are normalizing the augmented images using resnet preprocessing
x = Lambda(preprocess_input)(x)

# here we are passing the data through the base model
x = base_model(x, training=False)

# here we are adding the classifier head
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

# here we are creating the final model
model = Model(inputs, outputs)

# here we are compiling the model for stage 1
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# here we are starting the training for stage 1
print("starting model training stage 1 head only with advanced augmentations")
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS_STAGE_1,
    class_weight=class_weight_dict,
    verbose=1
)

# here we are finishing the training for stage 1
print("stage 1 training complete")


Building the ResNet50 model...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Freezing the ResNet50 base layers for Stage 1.
Applying advanced augmentations: Flip, Rotation, Zoom, and Contrast...

--- Starting Model Training (Stage 1: Head Only, WITH ADVANCED AUGMENTATIONS) ---
Epoch 1/15
     72/Unknown 25s 156ms/step - accuracy: 0.4309 - loss: 1.3649

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


72/72 ━━━━━━━━━━━━━━━━━━━━ 32s 252ms/step - accuracy: 0.4329 - loss: 1.3599 - val_accuracy: 0.7173 - val_loss: 0.7216
Epoch 2/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 14s 188ms/step - accuracy: 0.7292 - loss: 0.6612 - val_accuracy: 0.7801 - val_loss: 0.5750
Epoch 3/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 27s 380ms/step - accuracy: 0.7662 - loss: 0.5712 - val_accuracy: 0.7818 - val_loss: 0.6037
Epoch 4/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 14s 186ms/step - accuracy: 0.7832 - loss: 0.5241 - val_accuracy: 0.8237 - val_loss: 0.4746
Epoch 5/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 14s 187ms/step - accuracy: 0.8084 - loss: 0.4468 - val_accuracy: 0.8133 - val_loss: 0.5134
Epoch 6/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 20s 281ms/step - accuracy: 0.8119 - loss: 0.4472 - val_accuracy: 0.8237 - val_loss: 0.4857
Epoch 7/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 14s 194ms/step - accuracy: 0.8397 - loss: 0.4214 - val_accuracy: 0.8325 - val_loss: 0.4467
Epoch 8/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 14s 195ms/step - accuracy: 0.8398 - loss: 0.4084 - val_accuracy: 0.832

##Now we'll unfreeze the top 30 layers, re-compile with our low learning rate (1e-5), and continue training (passing the class weights again)

In [ ]:
# here we are importing tensorflow and the cosine decay scheduler
import tensorflow as tf
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.applications.resnet50 import preprocess_input

# here we are starting stage 2 fine tuning with optimized pipeline
print("starting model training stage 2 fine tuning optimized")

# here we are setting the number of training batches for the cosine decay schedule
NUM_TRAINING_BATCHES = 67

# here we are defining the cosine decay learning rate schedule
initial_learning_rate = 1e-4
decay_steps = (NUM_TRAINING_BATCHES * EPOCHS_STAGE_2)
lr_schedule = CosineDecay(
    initial_learning_rate,
    decay_steps=decay_steps,
    alpha=0.01
)

# here we are unfreezing the top 65 layers for deeper fine tuning
print("unfreezing the top 65 layers of the model...")
base_model.trainable = True
for layer in base_model.layers[:-65]:
    layer.trainable = False

# here we are recompiling the model with the cosine decay learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

# here we are confirming the model is recompiled
print("model recompiled with cosine decay learning rate schedule")
model.summary()

# here we are continuing the training for fine tuning
print("continuing training for deeper fine tuning for 10 epochs")
history_finetune_optimized = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=history.epoch[-1] + EPOCHS_STAGE_2,
    initial_epoch=history.epoch[-1],
    class_weight=class_weight_dict,
    verbose=1
)

# here we are finishing stage 2 fine tuning
print("stage 2 optimized fine tuning complete")



--- Starting Model Training (Stage 2: Fine-Tuning, OPTIMIZED) ---
Unfreezing the top 65 layers of the model...
--- Model Re-compiled with Cosine Decay LR Schedule ---


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast                 │ (None, 224, 224, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         8,196 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,595,908 (90.01 MB)

 Trainable params: 18,342,404 (69.97 MB)

 Non-trainable params: 5,253,504 (20.04 MB)

Continuing training for deeper fine-tuning (10 epochs)...
Epoch 15/24
72/72 ━━━━━━━━━━━━━━━━━━━━ 98s 325ms/step - accuracy: 0.8122 - loss: 0.4925 - precision: 0.8258 - recall: 0.7948 - val_accuracy: 0.8342 - val_loss: 0.6422 - val_precision: 0.8392 - val_recall: 0.8290
Epoch 16/24
72/72 ━━━━━━━━━━━━━━━━━━━━ 21s 294ms/step - accuracy: 0.9403 - loss: 0.1854 - precision: 0.9437 - recall: 0.9309 - val_accuracy: 0.8813 - val_loss: 0.4396 - val_precision: 0.8858 - val_recall: 0.8796
Epoch 17/24
72/72 ━━━━━━━━━━━━━━━━━━━━ 20s 282ms/step - accuracy: 0.9553 - loss: 0.1191 - precision: 0.9589 - recall: 0.9531 - val_accuracy: 0.9162 - val_loss: 0.3240 - val_precision: 0.9192 - val_recall: 0.9127
Epoch 18/24
72/72 ━━━━━━━━━━━━━━━━━━━━ 60s 840ms/step - accuracy: 0.9684 - loss: 0.0744 - precision: 0.9696 - recall: 0.9674 - val_accuracy: 0.9407 - val_loss: 0.2626 - val_precision: 0.9421 - val_recall: 0.9372
Epoch 19/24
72/72 ━━━━━━━━━━━━━━━━━━━━ 22s 300ms/step - accuracy: 0.9819 - loss: 0.0550 - prec

##Final Evaluation and Saving
This is our final experiment. We will see if our full pipeline (CV2 Preprocessing + Augmentation + Weighted Fine-Tuning) can finally beat our previous best score of 67.77%

In [ ]:
# here we are evaluating and saving the optimized fine tuned model
print("evaluating the optimized fine tuned model on the test set")
results = model.evaluate(test_dataset, verbose=1)

# here we are calculating all required metrics
metrics = {}
metrics['loss'] = results[0]
metrics['accuracy'] = results[1]
metrics['precision'] = results[2]
metrics['recall'] = results[3]

# here we are computing the f1 score
if (metrics['precision'] + metrics['recall']) > 0:
    metrics['f1_score'] = 2 * (metrics['precision'] * metrics['recall']) / (metrics['precision'] + metrics['recall'])
else:
    metrics['f1_score'] = 0.0

# here we are printing the test results
print("optimized fine tuned model test results")
print(f"test loss: {metrics['loss']:.4f}")
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"test precision: {metrics['precision']:.4f}")
print(f"test recall: {metrics['recall']:.4f}")
print(f"test f1 score: {metrics['f1_score']:.4f}")

# here we are saving the final model to google drive
print("saving the model to google drive")
os.makedirs("/content/drive/MyDrive/MODELS", exist_ok=True)
model.save("/content/drive/MyDrive/MODELS/resnet_final_optimized.h5")

# here we are confirming that the model has been saved and the experiment is complete
print("optimized fine tuned resnet model saved")
print("this completes the final optimization experiment")



--- Evaluating the OPTIMIZED Fine-Tuned Model on the Test Set ---
13/13 ━━━━━━━━━━━━━━━━━━━━ 22s 157ms/step - accuracy: 0.5785 - loss: 3.6105 - precision: 0.5862 - recall: 0.5785



--- OPTIMIZED Fine-Tuned Model Test Results ---
Test Loss: 1.7511
Test Accuracy: 0.7843
Test Precision: 0.7903
Test Recall: 0.7843
Test F1-Score: 0.7873

--- Saving the model to Google Drive ---
OPTIMIZED Fine-Tuned ResNet model saved.
This completes the final optimization experiment!
